# Q1 - The basic classifier

1. **Load:** the adult dataset.
2. **Preprocess:** clean missing values, drop useless columns and binarize age.
3. **Split:** split the dataset into a training and test set.
4. **Train:** a classifier that predicts if someone earns `>50K` 
5. **Measure:** test perofamance on test set using accuracy, precision, recall, and F1 score.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

RANDOM_STATE = 42
DATA_PATH = "../data/adult_with_headers.csv"

## 1. Loading the data

The raw file has two small mistakes that need to be fixed before it can be used:
- values are written with a leading space (`" Male"` instead of `"Male"`),
- missing values are written as `"?"` instead of being empty.

The loader fixes both while reading the file.

In [2]:
class AdultDatasetLoader:
    """Reads the Adult CSV file and returns it as a pandas DataFrame."""

    MISSING_VALUE_MARKER = "?"

    def __init__(self, csv_path):
        self.csv_path = csv_path

    def load(self):
        dataframe = pd.read_csv(
            self.csv_path,
            skipinitialspace=True,                 # removes the space after each comma
            na_values=self.MISSING_VALUE_MARKER,   # "?" becomes a real missing value (NaN)
        )
        return dataframe

In [3]:
loader = AdultDatasetLoader(DATA_PATH)
raw_data = loader.load()

print("Number of rows   :", raw_data.shape[0])
print("Number of columns:", raw_data.shape[1])
raw_data.head()

Number of rows   : 32561
Number of columns: 15


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [4]:
# How many missing values does each column have?
missing_per_column = raw_data.isna().sum()
missing_per_column[missing_per_column > 0]

workclass         1836
occupation        1843
native_country     583
dtype: int64

## 2. Preprocessing

Choices made (to justify in the report):

| Step | What we do | Why |
|---|---|---|
| Missing values | Drop the rows containing a `"?"` | Only about 7% of the rows, and it keeps things simple |
| `fnlwgt` | Drop the column | It is a census *sampling weight* (how many real people this row represents), not a property of the person |
| `education` | Drop the column | It holds the same information as `education_num` (e.g. `Bachelors` = 13) |
| `age` | Binarize: `1` if age >= 37, else `0` | Asked by the project. 37 is the median age, so both groups have about the same size |
| `sex` | Binarize: `1` = Male, `0` = Female | Makes it a simple 0/1 column, which Part 3 (fairness) and Part 4 (privacy) need |
| `income` (target) | `1` if `>50K`, else `0` | A classifier needs a number to predict |

The text columns (`workclass`, `occupation`, ...) are **not** turned into numbers here.
That happens inside the classifier (section 4), so the conversion only learns from the training data.

In [5]:
class AdultPreprocessor:
    """Cleans the raw Adult data and turns Age, Sex and the target into 0/1 columns."""

    TARGET_COLUMN = "income"
    POSITIVE_INCOME_LABEL = ">50K"
    COLUMNS_TO_DROP = ["fnlwgt", "education"]

    def __init__(self, age_threshold=37):
        # People with age >= age_threshold get the value 1 ("older"), the others 0 ("younger").
        self.age_threshold = age_threshold
        self.rows_before = None
        self.rows_after = None

    def preprocess(self, raw_dataframe):
        # We work on a copy so the original raw data is never modified.
        dataframe = raw_dataframe.copy()

        self.rows_before = len(dataframe)
        dataframe = self._drop_rows_with_missing_values(dataframe)
        self.rows_after = len(dataframe)

        dataframe = self._drop_unused_columns(dataframe)
        dataframe = self._binarize_age(dataframe)
        dataframe = self._binarize_sex(dataframe)
        dataframe = self._encode_target(dataframe)
        return dataframe

    def _drop_rows_with_missing_values(self, dataframe):
        dataframe = dataframe.dropna()
        # Renumber the rows 0, 1, 2, ... so the row numbers have no holes.
        dataframe = dataframe.reset_index(drop=True)
        return dataframe

    def _drop_unused_columns(self, dataframe):
        dataframe = dataframe.drop(columns=self.COLUMNS_TO_DROP)
        return dataframe

    def _binarize_age(self, dataframe):
        is_older = dataframe["age"] >= self.age_threshold
        dataframe["age"] = is_older.astype(int)
        return dataframe

    def _binarize_sex(self, dataframe):
        is_male = dataframe["sex"] == "Male"
        dataframe["sex"] = is_male.astype(int)
        return dataframe

    def _encode_target(self, dataframe):
        earns_more = dataframe[self.TARGET_COLUMN] == self.POSITIVE_INCOME_LABEL
        dataframe[self.TARGET_COLUMN] = earns_more.astype(int)
        return dataframe

    def print_report(self):
        rows_removed = self.rows_before - self.rows_after
        percentage_removed = 100 * rows_removed / self.rows_before
        print("Rows before cleaning:", self.rows_before)
        print("Rows after cleaning :", self.rows_after)
        print(f"Rows removed        : {rows_removed} ({percentage_removed:.1f}%)")
        print("Age rule            : 1 if age >=", self.age_threshold, "else 0")
        print("Sex rule            : 1 if Male, 0 if Female")

In [6]:
preprocessor = AdultPreprocessor(age_threshold=37)
data = preprocessor.preprocess(raw_data)

preprocessor.print_report()
data.head()

Rows before cleaning: 32561
Rows after cleaning : 30162
Rows removed        : 2399 (7.4%)
Age rule            : 1 if age >= 37 else 0
Sex rule            : 1 if Male, 0 if Female


,age,workclass,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,1,State-gov,13,Never-married,Adm-clerical,Not-in-family,White,1,2174,0,40,United-States,0
1,1,Self-emp-not-inc,13,Married-civ-spouse,Exec-managerial,Husband,White,1,0,0,13,United-States,0
2,1,Private,9,Divorced,Handlers-cleaners,Not-in-family,White,1,0,0,40,United-States,0
3,1,Private,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,1,0,0,40,United-States,0
4,0,Private,13,Married-civ-spouse,Prof-specialty,Wife,Black,0,0,0,40,Cuba,0


In [7]:
# Quick look at the groups created by binarization, and at the target.
print("Age groups (1 = 37 or older):")
print(data["age"].value_counts())
print()
print("Sex groups (1 = Male):")
print(data["sex"].value_counts())
print()
print("Target (1 = earns >50K):")
print(data["income"].value_counts())

Age groups (1 = 37 or older):
age
1    15572
0    14590
Name: count, dtype: int64

Sex groups (1 = Male):
sex
1    20380
0     9782
Name: count, dtype: int64

Target (1 = earns >50K):
income
0    22654
1     7508
Name: count, dtype: int64


Note: only about **25%** of the people earn `>50K`. So a "lazy" model that always answers `<=50K`
would already be right about 75% of the time. We will use that as a baseline to compare against.

## 3. Train / validation / test split

- **Train set (60%)**: the examples the model learns from.
- **Validation set (20%)**: used to choose the model settings (and later, in Part 3, the fairness fix).
- **Test set (20%)**: never touched until the very end, it tells us how good the model really is.

The split is **stratified**: each of the three sets keeps the same share of `>50K` people (about 25%).

The splitter **remembers which rows** went into which set. Part 4 asks to split the private
(noisy) data "in the same manner as in (1)", so we can then reuse exactly the same rows
with `apply_to(...)`.

In [8]:
class DatasetSplit:
    """Holds the three parts of a split: features (X) and target (y) for train, validation and test."""

    def __init__(self, X_train, y_train, X_validation, y_validation, X_test, y_test):
        self.X_train = X_train
        self.y_train = y_train
        self.X_validation = X_validation
        self.y_validation = y_validation
        self.X_test = X_test
        self.y_test = y_test

    def summary(self):
        rows = []
        parts = [
            ("train", self.y_train),
            ("validation", self.y_validation),
            ("test", self.y_test),
        ]
        for name, y in parts:
            rows.append({
                "set": name,
                "rows": len(y),
                "share of >50K": round(y.mean(), 3),
            })
        return pd.DataFrame(rows)


class DataSplitter:
    """Splits a dataset into train / validation / test and remembers which rows went where."""

    def __init__(self, target_column, validation_size=0.2, test_size=0.2, random_state=RANDOM_STATE):
        self.target_column = target_column
        self.validation_size = validation_size
        self.test_size = test_size
        self.random_state = random_state

        # Filled in by split(). They store the row numbers of each set.
        self.train_rows = None
        self.validation_rows = None
        self.test_rows = None

    def split(self, dataframe):
        all_rows = dataframe.index
        all_targets = dataframe[self.target_column]

        # Step 1: put the test rows aside.
        rest_rows, test_rows = train_test_split(
            all_rows,
            test_size=self.test_size,
            stratify=all_targets,
            random_state=self.random_state,
        )

        # Step 2: split what is left into train and validation.
        # validation_size is a share of the WHOLE dataset, so we rescale it
        # to a share of what is left (0.2 / 0.8 = 0.25).
        validation_share_of_rest = self.validation_size / (1 - self.test_size)
        train_rows, validation_rows = train_test_split(
            rest_rows,
            test_size=validation_share_of_rest,
            stratify=all_targets.loc[rest_rows],
            random_state=self.random_state,
        )

        self.train_rows = train_rows
        self.validation_rows = validation_rows
        self.test_rows = test_rows

        return self.apply_to(dataframe)

    def apply_to(self, dataframe):
        """Builds a DatasetSplit using the rows chosen by the last call to split()."""
        if self.train_rows is None:
            raise RuntimeError("Call split() at least once before apply_to().")

        features = dataframe.drop(columns=[self.target_column])
        target = dataframe[self.target_column]

        return DatasetSplit(
            X_train=features.loc[self.train_rows],
            y_train=target.loc[self.train_rows],
            X_validation=features.loc[self.validation_rows],
            y_validation=target.loc[self.validation_rows],
            X_test=features.loc[self.test_rows],
            y_test=target.loc[self.test_rows],
        )

In [9]:
splitter = DataSplitter(target_column="income", validation_size=0.2, test_size=0.2)
split = splitter.split(data)

split.summary()

,set,rows,share of >50K
0,train,18096,0.249
1,validation,6033,0.249
2,test,6033,0.249


## 4. The classifier

We use **logistic regression**. In short, it gives each column a weight, adds everything up,
and turns the total into a probability between 0 and 1 of earning `>50K`.
If the probability is 0.5 or more, the answer is `>50K`.

Why this model:
- it is **fast** to train, which helps Part 2 (we have to train it 100 times to measure energy);
- it gives **probabilities**, which Part 6 needs ("wrong but highly confident" means a high probability for the wrong answer);
- its weights are **easy to explain**, which helps Parts 6 and 7.

Before the model sees the data, the columns are converted into numbers:
- **text columns** (`workclass`, `occupation`, ...) get **one-hot encoded**: each possible value becomes its own 0/1 column
  (e.g. `occupation = Sales` becomes a column `occupation_Sales` that is 1 for salespeople and 0 for everybody else);
- **number columns** (`hours_per_week`, ...) get **scaled** to a similar range, so a column with big numbers
  (like `capital_gain`, up to 99999) does not dominate one with small numbers (like `education_num`, 1 to 16);
- **`age` and `sex`** are already 0/1, so they are kept as they are.

All of this is learned from the **training set only**, so no information leaks from the test set.

In [10]:
class IncomeClassifier:
    """Logistic regression that predicts if someone earns >50K (1) or not (0)."""

    CATEGORICAL_COLUMNS = [
        "workclass",
        "marital_status",
        "occupation",
        "relationship",
        "race",
        "native_country",
    ]
    NUMERICAL_COLUMNS = [
        "education_num",
        "capital_gain",
        "capital_loss",
        "hours_per_week",
    ]
    BINARY_COLUMNS = [
        "age",
        "sex",
    ]

    def __init__(self, regularization_strength=1.0):
        # Called "C" in scikit-learn. Smaller value = simpler model (weights pushed towards 0).
        self.regularization_strength = regularization_strength
        self.pipeline = self._build_pipeline()

    def _build_pipeline(self):
        column_converter = ColumnTransformer(
            transformers=[
                ("text_to_columns", OneHotEncoder(handle_unknown="ignore"), self.CATEGORICAL_COLUMNS),
                ("scale_numbers", StandardScaler(), self.NUMERICAL_COLUMNS),
                ("keep_binary", "passthrough", self.BINARY_COLUMNS),
            ],
            # Keep short column names like "occupation_Sales" instead of "text_to_columns__occupation_Sales".
            verbose_feature_names_out=False,
        )
        model = LogisticRegression(
            C=self.regularization_strength,
            max_iter=2000,
            random_state=RANDOM_STATE,
        )
        pipeline = Pipeline(
            steps=[
                ("convert_columns", column_converter),
                ("model", model),
            ]
        )
        return pipeline

    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self

    def predict(self, X):
        """Returns 0/1 answers."""
        return self.pipeline.predict(X)

    def predict_probability(self, X):
        """Returns, for every person, the probability of earning >50K."""
        probabilities_for_both_classes = self.pipeline.predict_proba(X)
        # Column 0 = probability of class 0 (<=50K), column 1 = probability of class 1 (>50K).
        return probabilities_for_both_classes[:, 1]

    def feature_weights(self):
        """Returns the weight the model gives to each (converted) column, biggest effect first."""
        column_names = self.pipeline.named_steps["convert_columns"].get_feature_names_out()
        weights = self.pipeline.named_steps["model"].coef_[0]

        table = pd.DataFrame({"feature": column_names, "weight": weights})
        table["absolute_weight"] = table["weight"].abs()
        table = table.sort_values("absolute_weight", ascending=False)
        table = table.drop(columns=["absolute_weight"])
        table = table.reset_index(drop=True)
        return table

## 5. Choosing the model setting on the validation set

Logistic regression has one main setting, `C` (the *regularization strength*).
- small `C` means a simpler model that may miss patterns (*underfitting*),
- large `C` means a model that may learn noise from the training data (*overfitting*).

We train one model per candidate value on the **train** set and compare them on the **validation** set.
We keep the value with the best validation accuracy. The test set is still untouched.

In [11]:
class HyperparameterTuner:
    """Tries several values of C and keeps the one with the best validation accuracy."""

    def __init__(self, candidate_values):
        self.candidate_values = candidate_values
        self.results = None
        self.best_value = None

    def tune(self, split):
        rows = []
        for value in self.candidate_values:
            classifier = IncomeClassifier(regularization_strength=value)
            classifier.fit(split.X_train, split.y_train)

            train_predictions = classifier.predict(split.X_train)
            validation_predictions = classifier.predict(split.X_validation)

            rows.append({
                "C": value,
                "train accuracy": accuracy_score(split.y_train, train_predictions),
                "validation accuracy": accuracy_score(split.y_validation, validation_predictions),
            })

        self.results = pd.DataFrame(rows)

        best_row_number = self.results["validation accuracy"].idxmax()
        self.best_value = self.results.loc[best_row_number, "C"]
        return self.best_value

In [12]:
tuner = HyperparameterTuner(candidate_values=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0])
best_C = tuner.tune(split)

print("Best C:", best_C)
tuner.results

Best C: 1.0


,C,train accuracy,validation accuracy
0,0.001,0.826315,0.826289
1,0.010,0.844607,0.848003
2,0.100,0.847701,0.852147
3,1.000,0.847425,0.854301
4,10.000,0.847535,0.853141
5,100.000,0.847701,0.853141


Now we train **the classifier** with the best `C`, on the training set.

(We do not merge train and validation here on purpose: Part 3 needs the validation set
to tune the fairness fix, so it has to stay "unseen" by the model.)

In [13]:
the_classifier = IncomeClassifier(regularization_strength=best_C)
the_classifier.fit(split.X_train, split.y_train)

## 6. Performance on the test set

The metrics, in plain words (">50K" is the "positive" answer):

- **Accuracy**: share of people for whom the answer is correct.
- **Precision**: among the people the model says earn `>50K`, the share who really do.
- **Recall**: among the people who really earn `>50K`, the share the model finds.
- **F1 score**: a single number that balances precision and recall (it is low if either one is low).
- **ROC AUC**: pick one random `>50K` person and one random `<=50K` person. This is the chance that the model gives
  the higher probability to the `>50K` one. 0.5 = random guessing, 1.0 = perfect.

We also show the **baseline**: a model that always answers `<=50K`.

In [14]:
class ClassifierEvaluator:
    """Computes the usual performance metrics of a classifier on a given set."""

    def __init__(self, classifier):
        self.classifier = classifier

    def compute_metrics(self, X, y_true):
        predictions = self.classifier.predict(X)
        probabilities = self.classifier.predict_probability(X)

        metrics = {
            "accuracy": accuracy_score(y_true, predictions),
            "precision": precision_score(y_true, predictions),
            "recall": recall_score(y_true, predictions),
            "f1 score": f1_score(y_true, predictions),
            "roc auc": roc_auc_score(y_true, probabilities),
        }
        return metrics

    def compute_baseline_accuracy(self, y_true):
        """Accuracy of a model that always answers 0 (<=50K)."""
        share_of_zeros = (y_true == 0).mean()
        return share_of_zeros

    def confusion_table(self, X, y_true):
        """Counts of right and wrong answers, shown as a readable table."""
        predictions = self.classifier.predict(X)
        counts = confusion_matrix(y_true, predictions)
        table = pd.DataFrame(
            counts,
            index=["really <=50K", "really >50K"],
            columns=["predicted <=50K", "predicted >50K"],
        )
        return table

    def metrics_table(self, X, y_true):
        metrics = self.compute_metrics(X, y_true)
        metrics["baseline accuracy (always <=50K)"] = self.compute_baseline_accuracy(y_true)

        table = pd.DataFrame.from_dict(metrics, orient="index", columns=["value"])
        table["value"] = table["value"].round(3)
        return table

In [15]:
evaluator = ClassifierEvaluator(the_classifier)

print("Performance of the classifier on the TEST set:")
evaluator.metrics_table(split.X_test, split.y_test)

Performance of the classifier on the TEST set:


,value
accuracy,0.849
precision,0.742
recall,0.607
f1 score,0.667
roc auc,0.903
baseline accuracy (always <=50K),0.751


In [16]:
evaluator.confusion_table(split.X_test, split.y_test)

,predicted <=50K,predicted >50K
really <=50K,4214,317
really >50K,591,911


### Which columns matter most?

Positive weight = pushes towards `>50K`, negative weight = pushes towards `<=50K`.

Be careful when reading this table:
- for a 0/1 column, the weight is the effect of going from 0 to 1;
  for a scaled number column, it is the effect of going up by one "typical spread" (one standard deviation).
  So the weights only give a rough ranking, not an exact comparison;
- some big weights come from **rare values** (e.g. a country with only a few dozen people),
  where the model has little data and the weight is not very reliable.

This is only a first look; Part 6 goes deeper into explanations.

In [17]:
the_classifier.feature_weights().head(15)

,feature,weight
0,capital_gain,2.482822
1,occupation_Priv-house-serv,-1.424660
2,native_country_Italy,1.258448
3,relationship_Wife,1.235804
4,relationship_Own-child,-1.235710
5,marital_status_Married-civ-spouse,1.178875
6,occupation_Farming-fishing,-1.115970
7,native_country_Columbia,-0.975720
8,relationship_Other-relative,-0.955070
9,native_country_South,-0.935456


Weights of the two protected attributes (useful context for Part 3):

In [18]:
all_weights = the_classifier.feature_weights()
protected_rows = all_weights["feature"].isin(["age", "sex"])
all_weights[protected_rows]

,feature,weight
12,age,0.820062
16,sex,0.803238


## Summary

- After cleaning we keep about 30,000 people; Age and Sex are now 0/1 columns.
- Split: 60% train / 20% validation / 20% test, stratified on the target, fixed seed (42).
- **The classifier** = logistic regression, `C` chosen on the validation set.
- Test results are in the tables above. The model clearly beats the "always `<=50K`" baseline.
  Its recall is lower than its precision: it misses a fair share of the people who really earn `>50K`.

Objects that the next parts can reuse: `data`, `splitter` (same rows via `splitter.apply_to(...)`),
`split`, `the_classifier`, `evaluator`.